# 🌍 Обучение и оценка: world model + VLM в MiniWorld

Полный прогон проекта на GPU: сбор данных → обучение RSSM → проверка глазами → таблица трёх методов → GIF и графики.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **GPU** (T4 достаточно). Запускать ячейки по порядку. Полный прогон ~15–25 минут (дольше всего — оценка VLM-метода: CLIP считается на каждом шаге планирования).

In [ ]:
%cd /content
!rm -rf project
!git clone -q -b develop https://github.com/RoGad/project-to-master-degree.git project
%cd project
!apt-get -qq install -y xvfb libglu1-mesa-dev libgl1-mesa-dev > /dev/null
!pip -q install gymnasium miniworld open_clip_torch imageio pyvirtualdisplay

## Виртуальный дисплей

MiniWorld рендерит через OpenGL, а у Colab нет монитора — поднимаем виртуальный (Xvfb). Без этой ячейки создание среды упадёт.

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1024, 768))
display.start()
print("Виртуальный дисплей запущен")

## Импорты и конфиг

Все гиперпараметры берутся из `src/config.py` репозитория — ноутбук ничего не переопределяет.

In [ ]:
import os, time
import numpy as np
import torch

import src.config as c
from src.utils import set_seed, save_checkpoint
from src.replay_buffer import ReplayBuffer, collect_random
from src.rssm import RSSM
from src.world_model import train_world_model, sample_batch, imagine_rollout
from src.clip_scorer import CLIPScorer
from src.evaluate import evaluate_all, record_gif, plot_success_rates, plot_training_loss
from src.environment import make_env, preprocess, to_image

set_seed()
print("device:", c.DEVICE, "| если тут cpu — включите GPU и перезапустите среду выполнения")

## 1. Сбор данных

Случайная политика наполняет буфер. Смотрим на долю успехов — это «пища» для reward-головы (baseline без VLM).

In [ ]:
t0 = time.time()
buf = collect_random(ReplayBuffer(), c.COLLECT_EPISODES)
print(f"эпизодов: {len(buf)} | переходов: {buf.num_transitions()} | "
      f"успехов случайной политики: {buf.num_successes()}/{len(buf)} | {time.time()-t0:.0f} c")

## 2. Обучение модели мира

Что смотреть в логе: **recon** должен заметно падать (модель учится видеть), **kl** держится около 3 (free nats), **reward** маленький. На T4 это несколько минут.

In [ ]:
wm = RSSM().to(c.DEVICE)
history = []
t0 = time.time()
train_world_model(wm, buf, history=history)
print(f"обучение заняло {time.time()-t0:.0f} c")
save_checkpoint(wm, "checkpoints/world_model.pt")
print("чекпоинт: checkpoints/world_model.pt")

## 3. Проверка глазами

Две картинки: реконструкции (модель «видит»?) и воображаемый rollout (модель «мечтает»?). Кадры будут размытыми — это нормально, важно чтобы комната, стены и красный ящик были различимы.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display as ipd
os.makedirs("results/plots", exist_ok=True)

# --- реконструкции: верх — реальные кадры, низ — как их видит модель ---
fr, pa, rw = sample_batch(buf)
with torch.no_grad():
    B, L = fr.shape[:2]
    emb = wm.encode(fr.reshape(B * L, 3, c.IMG_SIZE, c.IMG_SIZE)).reshape(B, L, -1)
    deter, stoch = wm.initial_state(B)
    for t in range(L):
        deter, stoch, _, _ = wm.obs_step(stoch, pa[:, t], deter, emb[:, t])
    recon = wm.decode(deter, stoch)
n = 6
fig, ax = plt.subplots(2, n, figsize=(n * 1.7, 3.6))
for i in range(n):
    ax[0, i].imshow(to_image(fr[i, -1])); ax[0, i].axis("off")
    ax[1, i].imshow(to_image(recon[i])); ax[1, i].axis("off")
plt.suptitle("Верх — реальные кадры, низ — реконструкция")
plt.tight_layout(); plt.savefig("results/plots/recon.png", dpi=110); plt.close()

# --- воображение: катим будущее только по prior и декодируем ---
env = make_env(); obs, _ = env.reset(seed=5); env.close()
with torch.no_grad():
    deter, stoch = wm.initial_state(1)
    zero = torch.zeros(1, c.NUM_ACTIONS, device=c.DEVICE)
    deter, stoch, _, _ = wm.obs_step(stoch, zero, deter, wm.encode(preprocess(obs)))
    seqs = torch.randint(0, c.NUM_ACTIONS, (1, c.HORIZON), device=c.DEVICE)
    states = imagine_rollout(wm, deter, stoch, seqs)
    dreams = [to_image(wm.decode(d, s)[0]) for d, s in states]
fig, ax = plt.subplots(1, len(dreams), figsize=(len(dreams) * 1.5, 1.9))
for i, im in enumerate(dreams):
    ax[i].imshow(im); ax[i].axis("off"); ax[i].set_title(f"+{i+1}", fontsize=8)
plt.suptitle("Воображаемый rollout (только prior)")
plt.tight_layout(); plt.savefig("results/plots/imagination.png", dpi=110); plt.close()

ipd(IPImage("results/plots/recon.png"))
ipd(IPImage("results/plots/imagination.png"))

## 4. Оценка: три метода

Главная таблица проекта: random / world-model planning (без VLM) / world-model planning + VLM. Несколько минут (VLM гоняет CLIP по воображаемым кадрам на каждом шаге).

In [ ]:
scorer = CLIPScorer()
t0 = time.time()
df = evaluate_all(wm, scorer)
df.to_csv("results/table.csv", index=False)
print(f"\n{len(c.EVAL_SEEDS)} сидов x {c.EVAL_EPISODES} эпизодов, лимит {c.EVAL_MAX_STEPS} шагов | {time.time()-t0:.0f} c")
df

## 5. GIF-визуализация

Записываем эпизоды под управлением VLM-планировщика (несколько сидов — показываем первый успешный) и один reward-эпизод для сравнения.

In [ ]:
shown = False
for seed in [77, 123, 5]:
    ok, n = record_gif(wm, "vlm", seed=seed, path=f"results/gifs/vlm_{seed}.gif", scorer=scorer)
    print(f"VLM   seed {seed}: success={ok}  шагов={n}")
    if ok and not shown:
        ipd(IPImage(f"results/gifs/vlm_{seed}.gif")); shown = True
ok, n = record_gif(wm, "reward", seed=77, path="results/gifs/reward_77.gif")
print(f"Reward seed 77: success={ok}  шагов={n}")

## 6. Графики для отчёта

In [ ]:
plot_success_rates(df, "results/plots/success_rates.png")
plot_training_loss(history, "results/plots/training_loss.png")
ipd(IPImage("results/plots/success_rates.png"))
ipd(IPImage("results/plots/training_loss.png"))

## 7. Скачать артефакты

Архив с чекпоинтом, таблицей, GIF и графиками — всё, что нужно для отчёта.

In [ ]:
!zip -qr artifacts.zip checkpoints results
from google.colab import files
files.download("artifacts.zip")

---
**Дальше:** артефакты сохраняем, финальные (таблица, лучший GIF, графики) осознанно добавим в репозиторий на этапе отчёта. Если таблица выглядит странно (например, VLM ≈ random) — не паникуем, разбираем: смотрим GIF (что делает агент), реконструкции (что видит CLIP) и обсуждаем.